In [1]:
!pip install wandb transformers peft trl datasets bitsandbytes


[notice] A new release of pip is available: 25.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import torch
import wandb
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments, 
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer


# 1. Connexion et Configuration W&B

# (Tu devras taper `wandb login` dans une cellule avec ton token W&B avant de lancer ça)
wandb.init(project="Mini-Projet-GameMaster", name="QLoRA-Run-1")

model_id = "Qwen/Qwen2.5-1.5B-Instruct"


# 2. Chargement du modèle avec QLoRA (Quantification 4-bit)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token 

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)


# 3. Configuration LoRA

lora_config = LoraConfig(
    r=16,               
    lora_alpha=32,      
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,  
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)


# 4. Chargement des Données (avec les nouveaux chemins !)

def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], 
        tokenize=False, 
        add_generation_prompt=False
    )
    return example

# Les chemins pointent bien vers le dossier datasets/
dataset = load_dataset("json", data_files={
    "train": "../datasets/train.json",
    "validation": "../datasets/validation.json"
}, field="examples")

dataset = dataset.map(apply_chat_template)


# 5. Hyper-paramètres

training_args = TrainingArguments(
    output_dir="../model_adapter/checkpoints", # Les sauvegardes vont dans model_adapter !
    eval_strategy="steps",
    eval_steps=20,                 
    save_strategy="steps",
    save_steps=20,                 
    logging_steps=5,               
    learning_rate=2e-4,            
    per_device_train_batch_size=2, 
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, 
    num_train_epochs=3,
    weight_decay=0.01,             
    load_best_model_at_end=True,   
    metric_for_best_model="eval_loss",
    report_to="wandb",             
    fp16=False,
    bf16=True                      
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=512,            
    tokenizer=tokenizer,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
)


# 6. Entraînement et Sauvegarde Finale

print("Début de l'entraînement...")
trainer.train()

print("Sauvegarde du modèle final dans model_adapter...")
# Le modèle final est sauvegardé proprement dans l'architecture
trainer.model.save_pretrained("../model_adapter/best_gm_adapter")
tokenizer.save_pretrained("../model_adapter/best_gm_adapter")
wandb.finish()


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("Chargement du modèle pour l'inférence...")

# 1. Recharger le modèle de base en 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

# 2. Charger le tokenizer et fusionner le modèle de base avec TON adaptateur
tokenizer = AutoTokenizer.from_pretrained("../model_adapter/best_gm_adapter")
model_fine_tuned = PeftModel.from_pretrained(base_model, "../model_adapter/best_gm_adapter")

# 3. Préparer une requête de test (tirée de tes exemples d'actions incohérentes)
messages = [
    {
        "role": "system", 
        "content": "Tu es le Game Master d'un jeu de rôle fantasy en solo. Tu proposes trois actions au joueur. Le joueur peut reformuler une action avec ses propres mots, mais une action qui ne correspond à aucune possibilité proposée doit être refusée."
    },
    {
        "role": "assistant", 
        "content": "Le grand dragon noir déploie ses ailes gigantesques et bloque l'entrée de la caverne en crachant des flammes. Vous pouvez:\nChoix A : Frapper ses écailles d'un coup d'épée bien placé.\nChoix B : Utiliser un bouclier pour parer le souffle ardent.\nChoix C : Chercher une issue secondaire sur les hauteurs."
    },
    {
        "role": "user", 
        "content": "Je sors mon fusil d'assaut M4 pour l'abattre."
    }
]

# 4. Formater et générer la réponse avec tes hyperparamètres de génération
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")

# max_new_tokens=250, top_p=0.8 et temperature=1.0 comme tu l'as défini
outputs = model_fine_tuned.generate(**inputs, max_new_tokens=250, temperature=1.0, top_p=0.8)

# Affichage du résultat en ignorant le prompt de départ
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("\n--- Réponse du Game Master ---")
print(response)

ModuleNotFoundError: No module named 'wandb'